In [1]:
print("hello world") 

hello world


In [2]:
# 1. Scaling 
# 2. Normalize 
# 3. transform
# 4. train & test CIFAR10 load 
# 5. dataloader 
# 6. Convoluational layer

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim 

import torchvision
from torchvision.transforms import transforms
from torchvision.datasets import MNIST 
from torch.utils.data import DataLoader 

In [20]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.5], std = [0.5]) 
])

In [21]:
train_set = MNIST("./dataa", train = True, download = True, transform = transform)
test_set = MNIST("./dataa", train = False, download = True, transform = transform)

In [22]:
trainloader = DataLoader(train_set, batch_size = 64, shuffle = True)
testloader = DataLoader(test_set, batch_size = 64) 

In [23]:
trainloader

In [24]:
# Image size =- 1x28x28 

In [25]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            # first Conv+ReLU & Maxpool
            nn.Conv2d(in_channels = 1, out_channels = 28, kernel_size = 3, stride = 1, padding = 1), 
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2), 

            # second Conv+ReLU & Maxpool 
            nn.Conv2d(in_channels = 28, out_channels = 64, kernel_size = 3, stride = 1, padding = 1), 
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2),

            # thrid Conv+ReLU & Maxpool 
            nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size = 3, stride = 1, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2)
        )

        self.fc_layers = nn.Sequential(
            # input layer 
            nn.Linear(in_features = 3*3*128, out_features = 250),
            nn.ReLU(),

            # hidden layer
            nn.Linear(in_features = 250, out_features = 350),
            nn.ReLU(),
            
            # output layer
            nn.Linear(in_features = 350, out_features = 10) 
        )

    def forward(self, x):
        output = self.conv_layers(x) 
        output = output.view(output.size(0), -1) # flatten 
        output = self.fc_layers(output) 

        return output 

In [26]:
model = CNN()

criteria  = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters()) 

In [27]:
criteria

CrossEntropyLoss()

In [29]:
# Training 

epochs = 10
model.train()

for epoch in range(epochs):
    epoch_training_loss = 0.0 

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criteria(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f" epoch {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)} ") 

 epoch 1/10 & loss = 0.17874567858926427 
 epoch 2/10 & loss = 0.04585563453489732 
 epoch 3/10 & loss = 0.032495685144519205 
 epoch 4/10 & loss = 0.027453787524834215 
 epoch 5/10 & loss = 0.02185430130821318 
 epoch 6/10 & loss = 0.01890763042802898 
 epoch 7/10 & loss = 0.016672001978337847 
 epoch 8/10 & loss = 0.013850702657780932 
 epoch 9/10 & loss = 0.010755845892111905 
 epoch 10/10 & loss = 0.014169975085046558 


In [34]:
from sklearn.metrics import accuracy_score, precision_score

y_true = []
y_pred = []

model.eval()

with torch.no_grad():
    for images, labels in testloader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

acc = accuracy_score(y_true, y_pred)
print("Accuracy:- ", acc*100)

pre = precision_score(y_true, y_pred, average='weighted')
print("Precision:- ", pre*100) 

Accuracy:-  99.13
Precision:-  99.13877332276093
